In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir("/content/drive/MyDrive/I1 RecSys")

# Interrogación 1 - Métricas de Recomendación



## Instrucciones Generales

0. Tienes 1 hora y 20 minutos para resolver y entregar esta evaluación
1. Está prohibido el uso de IA como chat GPT, Gemini o similares. Si se sorprende a alguien usándolo se le pedirá la prueba y se evaluará con 1,1.
2. Pueden realizar el práctico tanto en Google Colab como en un Jupyter Notebook local, pero no olvide entregar antes de que termine la hora de clases en este buzón. Debe entregar únicamente el archivo main.ipynb completo.
3. Esta actividad es individual, está prohibido conversar con compañeros para realizarla.
4. Sí está permitido el uso de código de los prácticos realizados en las ayudantías, tanto el que se te entrega por el equipo docente como el que tú mismo/a has realizado.

Esta evaluación puede realizarse tanto de manera local como usando Google Colab. En caso de realizarla localmente se requiere tener instalado Numpy. Además, se sugiere tener instalada la última versión de Python y de Numpy.

La evaluación consta de una **parte teórica** y una **parte práctica**, donde en esta última se deberá implementar las métricas solicitadas.



**NOMBRE ESTUDIANTE**: < colocar_nombre >

## Preguntas Teóricas (3pts)

En esta sección deberás contestar 3 preguntas teóricas

1. Explique con sus palabras la **diferencia entre Diversidad y Novedad** en sistemas de recomendación.
Incluir ¿Qué mide cada una? ¿Por qué una lista puede ser diversa pero no novedosa, o novedosa pero no diversa? ¿Cuál métrica (Diversidad o Novedad) es más importante en un sistema que busca recomendar productos de compra repetitiva? Justifique

**Respuesta**: -Coloque su respuesta en esta celda-

2. Considere dos listas de recomendación para un usuario:
- Lista A:  5 ítems muy populares entre todos los usuarios y muy similares entre sí
- Lista B: 5 ítems poco populares y de categorías muy distintas

a) ¿Cuál de las dos listas tendrá mayor diversidad según la métrica ILS (Intra-list Similarity)? Justifique.

b) ¿Cuál tendrá mayor novedad según la definición basada en log(1/popi)? Explique por qué.


**Respuesta**: -Coloque su respuesta en esta celda-

3. Compare las métricas MRR (Mean Reciprocal Rank), Precision@N (P@N) y MAP (Mean Average Precision). Indique para cada una:
¿Qué aspecto del rendimiento de un sistema de recomendación mide?
¿Cuál de estas métricas penaliza más fuertemente que los ítems relevantes aparezcan en posiciones bajas del ranking?

**Respuesta:** -Coloque su respuesta en esta celda-

## Implementeación de Métrica (7 pts)

En esta sección deberás implementar dos métricas: Mean Absolute Error (MAE) y Zero Proportion de nDCG. Para hacer la evaluación de esta métrica vamos a utilizar el algoritmo de *collaborative filtering* denominado **Sapling Similarity**, presentado en el paper [Sapling Similarity: a performing and interpretable memory-based tool for recommendation](https://arxiv.org/abs/2210.07039), aplicado al dataset MovieLens 100K. Este dataset contiene 100.000 tuplas (usuario, item, ranking), de las cuales 80.000 se utilizarán para entrenamiento y 20.000 para testear. Los rankings se encuentran en un rango de 1 a 5.

Contarán con dos instancias del modelo ya entrenados: Basada en Usuario y Basada en Items. No deben modificar el código del modelo ni del entrenamiento.

In [ ]:
from sapling_similarity import SaplingSimilarity
import numpy as np

In [ ]:
# modelo basado en usuarios
sapling_similarity_users_model = SaplingSimilarity.from_csv(
    train_csv="train.csv",
    test_csv="test.csv",
    user_column_name="user_id",
    item_column_name="movie_id",
    rating_column_name="rating",
    threshold=3,
    based_on="user"
)

sapling_similarity_users_model.train()

# modelo basado en items
sapling_similarity_items_model = SaplingSimilarity.from_csv(
    train_csv="train.csv",
    test_csv="test.csv",
    user_column_name="user_id",
    item_column_name="movie_id",
    rating_column_name="rating",
    threshold=3,
    based_on="item"
)

sapling_similarity_items_model.train()

/content/drive/MyDrive/I1 RecSys/sapling_similarity.py:129: RuntimeWarning: invalid value encountered in divide
  B = np.nan_to_num((1 - (co_ocurrences_matrix * (1 - co_ocurrences_matrix / items_interactions) + (items_interactions - co_ocurrences_matrix.T).T * (1 - (items_interactions - co_ocurrences_matrix.T).T / (
/content/drive/MyDrive/I1 RecSys/sapling_similarity.py:130: RuntimeWarning: invalid value encountered in divide
  number_of_items - items_interactions))).T / (items_interactions * (1 - items_interactions / number_of_items))).T * np.sign(((co_ocurrences_matrix * number_of_items / items_interactions).T / items_interactions).T - 1))


Los modelos entrenados cuenta con el método `test`, el cual retorna dos elementos:
- El primer elemento corresponde a una lista de tuplas con los **ratings reales** para las combinaciones usuarios e items presentes en el set de testeo, con la forma (user_id, item_id, rating_real).

- El segundo elemento corresponde a una lista de tuplas con los **ratings predichos** por el algoritmo para las combinaciones usuarios e items presentes en el set de testeo, con la forma (user_id, item_id, rating_predicted).

In [ ]:
y_test, y_pred_users = sapling_similarity_users_model.test()
y_test, y_pred_items = sapling_similarity_items_model.test()

Además, los modelos entrenados cuentan con el método `predict`, que retorna para un `user_id` específico las lista de tuplas (item_id, ranking) de los items que el usuario no ha evaluado. Con estas predicciones es posible rankear una lista de items para un usuario ordenando descendente los items según el rating que predice el modelo.

In [ ]:
sapling_similarity_users_model.predict(user_id=1)[40:50]

[(1, np.int64(613), np.float64(4.300253965152188)),
 (1, np.int64(285), np.float64(4.293830614167815)),
 (1, np.int64(173), np.float64(4.290822452377635)),
 (1, np.int64(641), np.float64(4.2713802820914895)),
 (1, np.int64(1111), np.float64(4.26898936283368)),
 (1, np.int64(357), np.float64(4.265864353132638)),
 (1, np.int64(513), np.float64(4.262441584025546)),
 (1, np.int64(1431), np.float64(4.26102251250427)),
 (1, np.int64(657), np.float64(4.261015377303021)),
 (1, np.int64(479), np.float64(4.260759452916226))]

Se les entrega un diccionario con las predicciones hechas por cada una de las versiones del algoritmo, donde cada llave es un user_id y tiene como valor una lista con tuplas (item, rating), donde rating es el valor que el modelo predice que el usuario asignará al item. Los elementos para cada usuario estan ordenadas por el rating que predijo el modelo.

In [ ]:
user_based_predictions = {user_id: sapling_similarity_users_model.predict(user_id) for user_id in sapling_similarity_users_model.user_map.keys()}
item_based_predictions = {user_id: sapling_similarity_items_model.predict(user_id) for user_id in sapling_similarity_items_model.user_map.keys()}

1. Implemente la métrica Mean Absolute Error (MAE) completando la función. Luego, use la función para reportar la métrica para las dos variaciones del modelo (basado en usuarios y basado en ítems).

In [ ]:
def mean_absolute_error(y_test, y_pred):
    """
    Calcula el error absoluto medio (MAE) entre las predicciones y los ratings reales.
    """
    ### RESPUESTA
    r_true = [r for _, _, r in y_test]
    r_pred = [r for _, _, r in y_pred]
    n = len(r_true)
    error = 0
    for r, r_hat in zip(r_true, r_pred):
        error += abs(r - r_hat)
    return error / n

In [ ]:
mean_absolute_error(y_test, y_pred_items)

np.float64(0.8282944995139453)

In [ ]:
mean_absolute_error(y_test, y_pred_users)

np.float64(0.808042290446764)

A continuación se crea un diccionario con los items relevantes para cada usuario presentes en `y_test`. Cada llave del diccionario es un user_id y tiene como valor un lista que contiene los item_ids de los items relevantes para ese usuario. Los ítems relevantes son aquellos a los que el usuario les asignó un **rating mayor o igual a 3**.

Ejemplo:
```python
{
  1: [3, 4, 8,...],
  2: [23, 24, 53,...],
  ...
}
```
donde [3, 4, 8,...] son los items relevantes para el usuario con ID = 1.

In [ ]:
relevant_items = {}

for user_id, item_id, rating in y_test:
    if user_id not in relevant_items:
        relevant_items[user_id] = []
    if rating >= 3:
        relevant_items[user_id].append(item_id)

2. Completa las funciones `dcg` y `ideal_dcg`, las cuales reciben como parámetros:

- `user_recommendations`:, Lista de tuplas (item_id, rating_predicted) de un usuario.

- `user_relevant_items` Lista de items_ids relevantes para el usuario.

- `k`: Largo de la lista de recomendación para calcular DCG.

La función `dcg` deberá retornar el DCG para un usuario en particular. Basate en la siguiente fórmula para implementarlo:

$$
DCG = \sum_{i=1}^n \frac{2^{\text{rel}_i} - 1}{\log_2(1 + i)}
$$
donde:
- $i$: el i-ésimo ítem de la lista de recomendación.
- $\text{rel}_i$: es una función indicatriz que indica si el i-ésimo item es relevante o no lo es.

---

**Ejemplo**

Supongamos que para un usuario tenemos:

`user_relevant_items = [A, B, D]`

`user_recommendations = [(A, 4.5), (B, 4.4), (C, 3.9), (D, 2.6), (E, 2.1)]`

Al realizar el cálculo para cada ítem quedaría:
- $i = 1$: $\frac{2^1 - 1}{\log_2 (1 + 1)}$
- $i = 2$: $\frac{2^1 - 1}{\log_2 (1 + 2)}$
- $i = 3$: $\frac{2^0 - 1}{\log_2 (1 + 3)}$
- $i = 4$: $\frac{2^1 - 1}{\log_2 (1 + 4)}$
- $i = 5$: $\frac{2^0 - 1}{\log_2 (1 + 5)}$

Se tendría para dicho usuario un DCG de:
$$
\sum_{i=1}^5 \frac{2^{\text{rel}_i} -1}{\log_2 (1+i)} = 1+0.63+0+0.43+0=2.06
$$

Por su parte, la función `ideal_dcg` deberá retornar el DCG de la lista de recomendación ordenada con los items relevantes en los primeros lugares.

In [ ]:
def dcg(user_recommendations, user_relevant_items, k):
    # COMPLETAR
    ### RESPUESTA
    dcg = 0
    for i, (item_id, rating) in enumerate(user_recommendations[:k]):
        if item_id in user_relevant_items:
            dcg += (2 ** 1 - 1) / np.log2(1 + (i + 1))
    return dcg

In [ ]:
def ideal_dcg(user_recommendations, user_relevant_items, k):
    # COMPLETAR
    ### RESPUESTA
    ideal_dcg = 0
    relevant_items = [item_id for item_id, rating in user_recommendations[:k] if item_id in user_relevant_items]
    for i in range(len(relevant_items)):
        ideal_dcg += (2 ** 1 - 1) / np.log2(1 + (i + 1))
    return ideal_dcg

3. Completa la función `zero_proportion_ndcg`, la cual debe entregar el porcentaje de usuarios que obtuvo un nDCG igual a 0. Esta función recibe como parámetros:
- `predictions_dict`: Diccionario en que cada llave es un user_id y su valor es una lista de tuplas `(user_id, item_id, rating_predicted)`.
- `relevant_items`: Diccionario en que cada llave es un user_id y su valor es una lista de items_ids relevantes para el usuario.
- `k`: Largo de la lista de recomendación para calcular Zero Proportion nDCG.

In [ ]:
def zero_proportion_ndcg(predictions_dict, relevant_items, k):
    """
    Calcula el porcentaje de usuarios con nDCG = 0.
    """
    total_ndcg = []
    n_zero_ndcg = 0

    for user_id, predictions in predictions_dict.items():
        if user_id in relevant_items:
            # Se convierten las tripletas (user_id, item_id, rating) a pares (item_id, rating)
            rec_list = []
            for _, item_id, rating in predictions:
                rec_list.append((int(item_id), float(rating)))

            # COMPLETAR
            ndcg_value = #

            # --------------------------------------------------------------------- RESPUESTA
            dcg_value = dcg(rec_list, relevant_items[user_id], k)
            ideal_dcg_value = ideal_dcg(rec_list, relevant_items[user_id], k)

            # Cuenta la cantidad de usuarios con nDCG igual a 0
            if dcg_value == 0 or ideal_dcg_value == 0:
                n_zero_ndcg += 1

            # Aborda el caso de si el denominador es 0
            if ideal_dcg_value > 0:
                ndcg_value = dcg_value / ideal_dcg_value
            else:
                ndcg_value = 0
            # ---------------------------------------------------------------------

            total_ndcg.append(ndcg_value)

    return n_zero_ndcg / len(total_ndcg)


4. Ejecute la siguiente celda que llama a la función `zero_proportion_ndcg` para cada modelo usando valores de k de 10, 100, 500, 1.000 y 2.000. Analice los resultados y mencione por qué podría producirse la diferencia de valores.

In [ ]:
ks = [10, 100, 500, 1000, 2000]

for k in ks:
    user_zero_proportion_ndcg = zero_proportion_ndcg(user_based_predictions, relevant_items, k)
    item_zero_proportion_ndcg = zero_proportion_ndcg(item_based_predictions, relevant_items, k)
    print(f"k={k}: user-based Zero Proportion nDCG={user_zero_proportion_ndcg:.4f}, item-based Zero Proportion nDCG={item_zero_proportion_ndcg:.4f}")

k=10: user-based Zero Proportion nDCG=0.9877, item-based Zero Proportion nDCG=0.6187
k=100: user-based Zero Proportion nDCG=0.1822, item-based Zero Proportion nDCG=0.1761
k=500: user-based Zero Proportion nDCG=0.0123, item-based Zero Proportion nDCG=0.0138
k=1000: user-based Zero Proportion nDCG=0.0046, item-based Zero Proportion nDCG=0.0046
k=2000: user-based Zero Proportion nDCG=0.0046, item-based Zero Proportion nDCG=0.0046
